In [47]:
import requests
import pandas as pd
import mysql.connector
from mysql.connector import Error
import numpy as np

In [48]:
url= 'https://beta.adalab.es/resources/apis/pelis/pelis.json'
datos= requests.get(url)
print (datos.status_code)
print (datos.content)

datos_pelis= datos.json()
print(datos_pelis)

200
b'[\n    {\n        "id": 1,\n        "titulo": "The Godfather",\n        "a\xc3\xb1o": 1972,\n        "duracion": 175,\n        "genero": "Crimen",\n        "adultos": false,\n        "subtitulos": [\n            "es",\n            "en"\n        ]\n    },\n    {\n        "id": 2,\n        "titulo": "The Godfather Part II",\n        "a\xc3\xb1o": 1974,\n        "duracion": 202,\n        "genero": "Crimen",\n        "adultos": false,\n        "subtitulos": [\n            "es",\n            "en"\n        ]\n    },\n    {\n        "id": 3,\n        "titulo": "Pulp Fiction",\n        "a\xc3\xb1o": 1994,\n        "duracion": 154,\n        "genero": "Crimen",\n        "adultos": true,\n        "subtitulos": [\n            "es",\n            "en"\n        ]\n    },\n    {\n        "id": 4,\n        "titulo": "Forrest Gump",\n        "a\xc3\xb1o": 1994,\n        "duracion": 142,\n        "genero": "Drama",\n        "adultos": false,\n        "subtitulos": [\n            "es",\n            

In [49]:
df_peliculas= pd.DataFrame(datos_pelis)
df_peliculas

,id,titulo,año,duracion,genero,adultos,subtitulos
0,1,The Godfather,1972,175,Crimen,False,"[es, en]"
1,2,The Godfather Part II,1974,202,Crimen,False,"[es, en]"
2,3,Pulp Fiction,1994,154,Crimen,True,"[es, en]"
3,4,Forrest Gump,1994,142,Drama,False,"[es, en, fr]"
4,5,The Dark Knight,2008,152,Acción,False,"[es, en]"
...,...,...,...,...,...,...,...
95,96,La vita è bella,1997,116,Drama,False,"[es, en, it]"
96,97,Requiem for a Dream,2000,102,Drama,True,"[es, en]"
97,98,Memento,2000,113,Thriller,True,"[es, en]"
98,99,Eternal Sunshine of the Spotless Mind,2004,108,Drama,False,"[es, en]"


In [50]:
try:
    conexion = mysql.connector.connect(   
        host = "127.0.0.1", 
        user= "root",
        password = "1522",  
       
    )
    print("Conexión exitosa")
except Error as e:
    print (f"Ha ocurrido un error:  {e}")

Conexión exitosa


In [51]:
try: 
    cursor= conexion.cursor()
    query = 'CREATE DATABASE IF NOT EXISTS Peliculas' 
    cursor.execute(query)
    print ('Query exitosa')
except Error as e:
    print(e)

Query exitosa


In [52]:
## estructuira de la tabla:
cursor.execute("USE Peliculas")
query = ''' CREATE TABLE IF NOT EXISTS peliculas_api (
    id INT PRIMARY KEY,
    titulo VARCHAR(255) NOT NULL,
    anio INT,
    duracion INT,
    genero VARCHAR(100),
    adultos BOOLEAN,
    subtitulos VARCHAR(255)

); '''

cursor.execute(query)
print("Tabla creada correctamente")

Tabla creada correctamente


In [53]:
df_peliculas.isnull().sum() ## confirmar si hay datos nulos

id            0
titulo        0
año           0
duracion      0
genero        0
adultos       0
subtitulos    0
dtype: int64

In [ ]:
query_insert = ''' INSERT INTO peliculas_api (
    id,
    titulo,
    anio,
    duracion,
    genero,
    adultos,
    subtitulos
)
VALUES (%s, %s, %s, %s, %s, %s, %s)'''

# Convertir subtítulos a texto
df_peliculas['subtitulos'] = df_peliculas['subtitulos'].astype(str)

# Convertir DataFrame a lista de listas
datos = df_peliculas.values.tolist()

# Insertar todos los datos
cursor.executemany(query_insert, datos)

# Guardar cambios
conexion.commit()

print(f"Se han insertado {cursor.rowcount} películas")

In [56]:
## fase 4 
## ¿Cuántas películas tienen una duración superior a 120 minutos?
query_duracion='''select count(*) from peliculas_api    
where duracion > 120;''' 
cursor.execute(query_duracion)
resultado = cursor.fetchone()

print(resultado)

(59,)


In [57]:
## ¿Cuántas películas incluyen subtítulos en español?
query_espaniol= '''select count(*) from peliculas_api
where subtitulos like '%es%';'''
cursor.execute(query_espaniol)
resultado = cursor.fetchone()

print(resultado)

(100,)


In [59]:
## ¿Cuántas películas tienen contenido adulto?
query_adultos= '''select count(*) from peliculas_api
where adultos like '1';'''
cursor.execute(query_adultos)
resultado = cursor.fetchone()

print(resultado)

(47,)


In [61]:
## ¿Cuál es la película más antigua registrada en la base de datos?

query_antigua='''select min(anio)
from peliculas_api;'''
cursor.execute(query_antigua)
resultado = cursor.fetchone()

print(resultado)

(1941,)


In [ ]:
## Muestra el promedio de duración de las películas agrupado por género.
query_promedio= '''select genero, avg(duracion) from peliculas_api
group by genero;'''
cursor.execute(query_promedio)
resultado = cursor.fetchall()
# Convertir a DF para ver los datos
df_resultado = pd.DataFrame(resultado, columns=['genero', 'promedio_duracion'])


df_resultado

,genero,promedio_duracion
0,Crimen,154.2857
1,Drama,126.2593
2,Acción,139.4444
3,Ciencia ficción,136.3077
4,Romance,139.6667
5,Bélico,161.0000
6,Thriller,121.6667
7,Musical,128.0000
8,Fantasía,159.8000
9,Aventura,133.0000


In [67]:
## ¿Cuántas películas por año se han registrado en la base de datos? Ordena de mayor a menor.
query_anios= '''select anio, count(*) as total_pelis  
from peliculas_api
group by anio
order by total_pelis desc;'''

cursor.execute(query_anios)
resultado = cursor.fetchall()

df_total_pelis = pd.DataFrame(resultado, columns=['genero', 'promedio_duracion'])


df_total_pelis

,genero,promedio_duracion
0,2001,5
1,2013,4
2,1994,4
3,2008,4
4,1999,4
5,2017,4
6,2010,3
7,1998,3
8,2014,3
9,2000,3


In [68]:
## ¿Cuál es el año con más películas en la base de datos?
query_anio_maspelis= '''select anio, count(*) as total_pelis
from peliculas_api
group by anio
order by total_pelis desc 
limit 1;'''
cursor.execute(query_anio_maspelis)
resultado = cursor.fetchone()

print(resultado)

(2001, 5)


In [70]:
## Obtén un listado de todos los géneros y cuántas películas corresponden a cada uno.
query_gen= '''select genero, count(*) total_pelis
from peliculas_api
group by genero
order by total_pelis desc;'''

cursor.execute(query_gen)
resultado = cursor.fetchall()

df_genero = pd.DataFrame(resultado, columns=['genero', 'peliculas'])


df_genero

,genero,peliculas
0,Drama,27
1,Ciencia ficción,13
2,Acción,9
3,Animación,9
4,Crimen,7
5,Terror,7
6,Thriller,6
7,Fantasía,5
8,Romance,3
9,Aventura,3


In [71]:
## Muestra todas las películas cuyo título contenga la palabra "Godfather" (puedes usar cualquier palabra).
query_titulo='''select * from peliculas_api
where titulo like '%Godfather%';'''

cursor.execute(query_titulo)

resultado = cursor.fetchall()

df_titulo = pd.DataFrame(resultado,
    columns=[
        'id',
        'titulo',
        'anio',
        'duracion',
        'genero',
        'adultos',
        'subtitulos'])

df_titulo

,id,titulo,anio,duracion,genero,adultos,subtitulos
0,1,The Godfather,1972,175,Crimen,0,"['es', 'en']"
1,2,The Godfather Part II,1974,202,Crimen,0,"['es', 'en']"
